# 10M Data Processing Pipeline
## E-Commerce Dress Sales Analytics — Large-Scale Data Architecture

This notebook explains and demonstrates the end-to-end data processing pipeline used to handle **10,000,000 synthetic transaction records** without loading the full raw dataset into memory during live web application execution.

---
## 1. Architecture Rationale: Why Full 10M Data is NOT Loaded Directly

In production business intelligence and scalable analytics engineering, web dashboards never directly parse multi-gigabyte transactional datasets into memory. Loading 10,000,000 raw rows on demand creates severe operational bottlenecks:

- **Prevents RAM Overload:** A raw 10M transaction CSV is approximately 1–2 GB on disk and easily expands to 4–8 GB of RAM when parsed into Pandas DataFrames. Web application servers and cloud containers running on standard memory tiers (512 MB – 2 GB) crash immediately with Out-Of-Memory (OOM) errors.
- **Keeps Dashboard Fast:** Reading and parsing 10,000,000 records on server startup takes 30–120 seconds, and live multi-column group-by aggregations on every page request induce heavy latency. By pre-aggregating metrics offline, API responses are delivered in **under 50 milliseconds**.
- **Follows Scalable Analytics Practice:** In enterprise architectures (Snowflake, BigQuery, Databricks), transactional raw logs are stored in cloud data lakes and processed via batch ETL pipelines into structured, indexed data marts. The user-facing dashboard queries only the optimized summary tables.

---
## 2. End-to-End Data Pipeline Flow

The project decouples offline big data processing from online real-time dashboard presentation:

```
Original dataset structure (data/dress_sales.csv: 10,001 rows, 18 columns)
        ↓
10M transaction generation (retaining empirical distributions & domain vocabulary)
        ↓
Chunk-wise processing (100,000 rows per batch — streaming generation & aggregation)
        ↓
Dashboard summary output (data/processed/dashboard_summary_10M.json: ~22 KB)
        ↓
Customer aggregation (data/processed/customer_agg_10M.csv: 500,000 accounts)
        ↓
Prediction sample (data/processed/prediction_sample_10M.csv: 4,500 points)
        ↓
FastAPI backend (reads pre-aggregated summary files in milliseconds)
        ↓
React dashboard (renders interactive KPI cards, charts, ledgers & forecasting)
```

---
## 3. Google Colab Execution Workflow

The 10M generation and chunk-wise aggregation pipeline is designed to execute in **Google Colab** with **Google Drive** mounted for cloud persistence.

The code cells below demonstrate the exact pipeline workflow used to produce the production-ready analytical outputs.

In [ ]:
# Step 1: Mount Google Drive for persistent cloud storage
from google.colab import drive
import os

drive.mount('/content/drive')

GDRIVE_DIR = "/content/drive/MyDrive/Ecommerce_Dress_Sales_10M"
os.makedirs(f"{GDRIVE_DIR}/raw", exist_ok=True)
os.makedirs(f"{GDRIVE_DIR}/processed", exist_ok=True)
print("Google Drive storage paths ready:")
print(f"  Raw cloud archive: {GDRIVE_DIR}/raw")
print(f"  Processed outputs: {GDRIVE_DIR}/processed")

In [ ]:
# Step 2: Clone repository in Google Colab environment
!git clone https://github.com/tithisultana/E-commerce_dress_sales_analytics.git
%cd E-commerce_dress_sales_analytics
!git checkout feature/10m-company-dashboard
!ls -la

In [ ]:
# Step 3: Install required data science and ML dependencies
!pip install --quiet pandas numpy scikit-learn tqdm

### Executing Chunk-Wise Data Processing
The script `scripts/generate_10m_processed_data.py` implements the memory-safe streaming logic:
1. Generates synthetic records in **100 iterations of 100,000 rows**.
2. Dynamically updates in-memory accumulators (KPIs, monthly totals, category shares, location distributions, and customer spending).
3. Releases each 100,000-row batch from RAM immediately to maintain peak memory usage under ~400 MB.
4. Performs `MiniBatchKMeans(k=3)` clustering on 500,000 customer accounts.
5. Writes the raw compressed file (`dress_sales_10M.csv.gz`) to Google Drive and the 3 processed summaries to `data/processed/`.

In [ ]:
# Step 4: Execute the chunk-wise 10M processing script
# (In Colab, this processes all 10,000,000 rows in ~5-15 minutes)
!python scripts/generate_10m_processed_data.py

In [ ]:
# Step 5: Check and verify generated output files
import os

PROCESSED_DIR = "data/processed"
print("Verifying generated files in", PROCESSED_DIR, ":")
for fname in sorted(os.listdir(PROCESSED_DIR)):
    fpath = os.path.join(PROCESSED_DIR, fname)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fname:<32} {size_kb:>10.2f} KB")

---
## 4. Output File Descriptions

The pipeline outputs three optimized analytical artifacts used by the web backend:

### 1. `dashboard_summary_10M.json` (~22 KB)
- **Role:** Powers the Overview KPIs, Sales Analytics charts, Product rankings, and Territory breakdown.
- **Structure & Contents:**
  - `kpis`: Total Net Revenue (৳62,632,086,622), Total Orders (10,000,000), Total Customers (500,000), Total Units Sold (30,000,746), Average Order Value (৳6,263), Average Rating (3.75).
  - `date_range`: Multi-year period spanning 2023-01-01 to 2026-12-31.
  - `monthly`: 48 consecutive months of pre-aggregated revenue, order counts, and units.
  - `categories`: Net revenue, units, and average rating across all dress categories (Saree, Dress, Kurti, etc.).
  - `payments`: Transaction revenue and percentage share for Rocket, bKash, Card, Cash on Delivery, and Nagad.
  - `locations`: Regional revenue, fulfillment units, and buyer volume across districts (Dhaka, Chattogram, Sylhet, etc.).
  - `products_top30`: Top 30 bestselling products ranked by total units fulfilled.
  - `cluster_summary`: Centroid averages (spending, orders, items) for the 3 behavioral customer segments.
  - `customer_stats`: Overall buyer counts and repeat customer frequency.
  - `methodology`: Dataset dimensions and revenue accounting formulas.

### 2. `customer_agg_10M.csv` (~21 MB, 500,000 rows)
- **Role:** Powers the Customer Analytics ledger and full customer profile lookup.
- **Columns:** `Customer_ID`, `spend`, `items`, `orders`, `age`, `rating`, `location`, `cluster`.
- **Benefit:** Eliminates dynamic aggregation across 10M rows. The backend performs fast, indexed pagination directly on the pre-computed 500k customer profiles.

### 3. `prediction_sample_10M.csv` (~100 KB, 4,500 rows)
- **Role:** Powers the interactive Customer Behavioral Scatter Plot in the React frontend.
- **Columns:** `Customer_ID`, `spend`, `orders`, `cluster`.
- **Benefit:** Attempting to render 500,000 scatter points in SVG/Canvas freezes client browser tabs. A balanced stratified sample of 1,500 accounts per cluster (4,500 points total) perfectly illustrates the cohort distribution while maintaining 60 FPS client rendering.

In [ ]:
# Quick programmatic inspection of the generated files
import json
import pandas as pd

# 1. Summary JSON inspection
with open("data/processed/dashboard_summary_10M.json", "r", encoding="utf-8") as f:
    summary = json.load(f)
print("=== 1. Summary KPIs ===")
for k, v in summary["kpis"].items():
    print(f"  {k:<22}: {v}")

# 2. Customer Aggregation preview
customer_agg = pd.read_csv("data/processed/customer_agg_10M.csv", nrows=3)
print("\n=== 2. Customer Aggregation (Top 3) ===")
print(customer_agg.to_string(index=False))

# 3. Prediction Scatter Sample preview
pred_sample = pd.read_csv("data/processed/prediction_sample_10M.csv", nrows=3)
print("\n=== 3. Prediction Sample (Top 3) ===")
print(pred_sample.to_string(index=False))

---
## 5. Notebook Ecosystem & Context Within the Project

This project organizes data science and analytical modeling into a structured, modular workflow:

1. **Notebooks 01–03 (Exploratory Analysis & Model Prototyping):**
   - **`01_E-Commerce_Dress_Sales_EDA.ipynb`:** Focuses on initial Exploratory Data Analysis, KPI definitions, temporal patterns, discount effects, and category breakdowns on the base dataset.
   - **`02_E-Commerce_Dress_Sales_Prediction.ipynb`:** Establishes the demand forecasting regression pipeline, feature transformations (SimpleImputer + OneHotEncoder), train/test holdout validation, and algorithmic comparison (Linear Regression baseline vs. Random Forest Regressor).
   - **`03_E-Commerce_Customer_Segmentation.ipynb`:** Formulates customer behavioral grouping using standardized gross spend, order frequency, and volume with K-Means (k=3) clustering.

2. **Notebook 04 (Production Scaling & 10M Data Pipeline):**
   - **`04_10M_Data_Processing_Pipeline.ipynb` (This Notebook):** Bridges the gap between standalone data science experimentation and high-performance production dashboard deployment. It operationalizes the analytical discoveries of Notebooks 01–03 at 10,000,000 scale using cloud archiving (Google Drive) and chunk-wise batch processing to supply the live FastAPI + React application with lightweight, instant-serving analytical outputs.